# DFU-RadialAdapter — GitHub-first, persistence-safe pilot
This notebook verifies Google Drive with stale-mount repair and a real write/read check before training. It refuses to train unless Drive and the `GITHUB_TOKEN` secret are available.

In [ ]:
# DFU-RadialAdapter GitHub-first pilot — one executable cell
import os
import sys
import json
import time
import uuid
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

MOUNT_POINT = Path("/content/drive")
MY_DRIVE = MOUNT_POINT / "MyDrive"

def _is_mountpoint(path: Path) -> bool:
    try:
        return subprocess.run(
            ["mountpoint", "-q", str(path)],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        ).returncode == 0
    except Exception:
        return False

def _verify_drive_read_write() -> Path:
    if not MY_DRIVE.is_dir():
        raise RuntimeError(f"{MY_DRIVE} is unavailable after mounting.")

    check_dir = MY_DRIVE / "DFU-ImageGuard" / "_mount_verification"
    check_dir.mkdir(parents=True, exist_ok=True)
    check_file = check_dir / f"verified_{uuid.uuid4().hex}.txt"
    expected = f"DFU-ImageGuard verified at {time.time_ns()}"

    check_file.write_text(expected, encoding="utf-8")
    observed = check_file.read_text(encoding="utf-8")
    if observed != expected:
        raise RuntimeError("Google Drive write/read verification did not match.")
    check_file.unlink(missing_ok=True)
    return check_dir.parent

def _clean_stale_mount() -> None:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass

    time.sleep(2)

    if _is_mountpoint(MOUNT_POINT):
        try:
            subprocess.run(
                ["fusermount", "-uz", str(MOUNT_POINT)],
                check=False,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
        except Exception:
            pass
        time.sleep(1)

    if not _is_mountpoint(MOUNT_POINT):
        try:
            shutil.rmtree(MOUNT_POINT, ignore_errors=True)
        except Exception:
            pass
        MOUNT_POINT.mkdir(parents=True, exist_ok=True)

def mount_drive_verified(max_attempts: int = 3) -> Path:
    errors = []

    # Reuse a healthy existing mount without opening another auth dialog.
    if MY_DRIVE.is_dir():
        try:
            project_root = _verify_drive_read_write()
            print(f"Google Drive already mounted and verified: {project_root}")
            return project_root
        except Exception as exc:
            errors.append(f"existing mount verification: {type(exc).__name__}: {exc}")

    for attempt in range(1, max_attempts + 1):
        force = attempt > 1
        try:
            if force:
                print(f"Repairing stale Drive mount before attempt {attempt}/{max_attempts}...")
                _clean_stale_mount()

            print(f"Mounting Google Drive — attempt {attempt}/{max_attempts}...")
            drive.mount(str(MOUNT_POINT), force_remount=force)

            project_root = _verify_drive_read_write()
            print(f"Google Drive mounted and write-verified: {project_root}")
            return project_root
        except Exception as exc:
            errors.append(f"attempt {attempt}: {type(exc).__name__}: {exc}")
            print(f"Drive attempt {attempt} failed: {exc}")
            time.sleep(2)

    details = "\n".join(f"- {item}" for item in errors)
    raise RuntimeError(
        "Google Drive could not be mounted after verified retries. "
        "Training was NOT started and no result was lost.\n\n"
        "Do these once, then reopen this notebook link:\n"
        "1. Runtime → Disconnect and delete runtime\n"
        "2. Allow pop-ups and third-party cookies for colab.research.google.com\n"
        "3. Sign in to the Google account whose MyDrive will hold the checkpoints\n"
        "4. Run the single cell and approve the Drive permission dialog\n\n"
        f"Recorded errors:\n{details}"
    )

project_root = mount_drive_verified()

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError(
        "Add GITHUB_TOKEN in Colab Secrets, enable notebook access, then rerun. "
        "Training was NOT started."
    )
os.environ["GITHUB_TOKEN"] = token

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm>=1.0.9",
        "kagglehub>=0.3",
        "ImageHash>=4.3",
        "scikit-learn>=1.5",
        "scipy>=1.13",
        "matplotlib>=3.9",
        "pandas>=2.2",
        "Pillow>=10.4",
        "tabulate>=0.9",
    ],
    check=True,
)

REPO = "https://github.com/AzizulHakim00/DFU-ImageGuard.git"
SOURCE_BRANCH = "radial-adapter-pilot-v1"
WORK = Path("/content/DFU-ImageGuard-radial")

if WORK.exists():
    shutil.rmtree(WORK)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", SOURCE_BRANCH, REPO, str(WORK)],
    check=True,
)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.radial_pilot_runner import PilotSettings, run_radial_adapter_pilot

settings = PilotSettings(
    run_id="RADIAL_ADAPTER_PILOT_V1",
    drive_root=str(project_root),
    secondary_drive_root="/content/drive/MyDrive/DFU-ImageGuard-Backup",
    seeds=(2026, 2027),
    outer_fold=0,
    max_epochs=25,
    patience=7,
    batch_size=16,
    num_workers=2,
    github_export=True,
    github_export_required=True,
    github_branch="radial-pilot-results",
    github_chunk_full_checkpoints=True,
    github_chunk_bytes=48 * 1024 * 1024,
    github_export_after_each_trial=True,
    require_secondary_drive_backup=True,
)

result = run_radial_adapter_pilot(settings=settings, repository_dir=WORK)
print(json.dumps(result, indent=2, default=str))
